# Chapter 8 — Tree-Based Algorithms and Ensemble Methods

**Nama:** Omar Al Farizy Hutagalung 

**NIM:** 101032300034  

**Mata Kuliah:** Machine Learning  

**Tugas:** Tugas 2 — Code Reproduction & Theoretical Deep-Dive

---

## Objective

Chapter ini membahas decision tree, random forest, bagging, gradient boosting, tuning hyperparameter, dan feature importance.

## Theory — Decision Trees

Decision tree membagi data berdasarkan fitur agar setiap kelompok hasil split
menjadi lebih homogen. Kriteria split dapat menggunakan Gini impurity atau
entropy. Pohon yang terlalu dalam dapat overfit, sehingga parameter seperti
`max_depth` dan `min_samples_leaf` digunakan untuk mengontrol kompleksitas.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)
tree_pred = tree.predict(X_test)
print("Decision Tree accuracy:", round(accuracy_score(y_test, tree_pred), 4))
print(classification_report(y_test, tree_pred, target_names=iris.target_names))

plt.figure(figsize=(12, 7))
plot_tree(tree, feature_names=iris.feature_names,
          class_names=iris.target_names, filled=True, rounded=True)
plt.title("Decision Tree — Iris")
plt.show()

## Theory — Random Forest and Bagging

Bagging melatih beberapa model pada sampel data yang berbeda, kemudian
menggabungkan prediksinya. Random Forest juga memilih subset fitur secara
acak pada split pohon, sehingga mengurangi korelasi antar pohon dan sering
mengurangi variance dibanding satu decision tree.

In [ ]:
forest = RandomForestClassifier(
    n_estimators=300, max_features="sqrt",
    random_state=42, n_jobs=-1
)
forest.fit(X_train, y_train)
forest_pred = forest.predict(X_test)

print("Random Forest accuracy:", round(accuracy_score(y_test, forest_pred), 4))
print(classification_report(y_test, forest_pred, target_names=iris.target_names))

importance = pd.Series(
    forest.feature_importances_, index=iris.feature_names
).sort_values(ascending=False)
print(importance)

importance.sort_values().plot(kind="barh", title="Random Forest Feature Importance")
plt.xlabel("Importance")
plt.show()

## Theory — Gradient Boosting

Gradient Boosting membangun model secara berurutan. Model berikutnya berusaha
memperbaiki kesalahan model sebelumnya. `learning_rate` mengatur kontribusi
setiap tahap, sedangkan `n_estimators` menentukan jumlah tahap boosting.

In [ ]:
gbm = GradientBoostingClassifier(
    n_estimators=100, learning_rate=0.1,
    max_depth=2, random_state=42
)
gbm.fit(X_train, y_train)
gbm_pred = gbm.predict(X_test)

print("Gradient Boosting accuracy:", round(accuracy_score(y_test, gbm_pred), 4))
print(classification_report(y_test, gbm_pred, target_names=iris.target_names))

## Theory — Hyperparameter Tuning and Model Comparison

GridSearchCV memilih kombinasi hyperparameter melalui cross-validation.
Perbandingan model perlu menggunakan data split dan metrik yang sama.
Test set tidak digunakan untuk memilih parameter.

In [ ]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 3, 5],
    "min_samples_leaf": [1, 2, 4]
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid, cv=cv, scoring="accuracy", n_jobs=-1
)
search.fit(X_train, y_train)
tuned_pred = search.predict(X_test)

print("Best parameters:", search.best_params_)
print("Best CV accuracy:", round(search.best_score_, 4))
print("Tuned Random Forest test accuracy:", round(
    accuracy_score(y_test, tuned_pred), 4
))

comparison = pd.DataFrame([
    {"model": "Decision Tree", "test_accuracy": accuracy_score(y_test, tree_pred)},
    {"model": "Random Forest", "test_accuracy": accuracy_score(y_test, forest_pred)},
    {"model": "Gradient Boosting", "test_accuracy": accuracy_score(y_test, gbm_pred)},
    {"model": "Tuned Random Forest", "test_accuracy": accuracy_score(y_test, tuned_pred)}
]).sort_values("test_accuracy", ascending=False)
comparison

## Analysis and Conclusion

- Decision Tree mudah divisualisasikan tetapi dapat overfit.
- Random Forest menggabungkan banyak tree untuk mengurangi variance.
- Gradient Boosting membangun model secara berurutan untuk memperbaiki kesalahan.
- Feature importance membantu melihat kontribusi fitur, tetapi tidak membuktikan hubungan sebab-akibat.

**Conclusion:** Tree-based algorithms menawarkan trade-off antara interpretabilitas
dan performa. Evaluasi dan tuning perlu dilakukan secara konsisten agar
perbandingan model dapat dipercaya.